# ITDA 3rd 학술제 - 소비기한 OCR 추론 노트북
팀: `DScover_카피바라` (itda3-dscover-capybara)

아키텍처 요약은 팀 저장소의 `[DScover_카피바라]_아키텍처구조도.pdf` 및 `README.md` 를 참고하세요.


In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


## 1. 환경 설정
- CPU 전용, 오프라인 실행을 전제로 합니다 (`gpu=False`, `download_enabled=False`).
- 가중치는 `./weights` 폴더에 사전 배치되어 있어야 합니다 (`download_weights.sh` 참고).
- 4-Core vCPU 채점 환경을 가정하여 `ThreadPoolExecutor` 로 이미지 단위 병렬처리를 수행합니다.
  (Jupyter 커널 내부에서 `multiprocessing.Pool(fork)` 를 쓰면, 커널이 이미 여러 백그라운드
  스레드를 띄운 상태이므로 fork 직후 자식 프로세스가 락을 획득하지 못해 멈추는 경우가 있어
  스레드 기반 병렬화로 대체했습니다.)


In [ ]:
import os, sys, time, re, glob, warnings, threading
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

# 각 스레드에서 torch/BLAS가 내부적으로 다시 멀티스레드를 켜서 서로 경쟁(oversubscription)하는
# 것을 막기 위해, 프로세스 전체의 BLAS 스레드 수를 1로 고정한다. (워커 스레드 수만큼만 코어 활용)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# 일부 numpy/scipy 조합에서는 numpy.testing 모듈이 자체적으로 _blas_supports_fpe 심볼을
# 참조하지만 실제 컴파일된 확장에는 없어 scipy.ndimage import 시점에 크래시가 나는 경우가
# 있다(OCR 기능과 무관한 numpy 내부 테스트 유틸리티 심볼). easyocr이 내부적으로 scipy.ndimage
# 를 import 하므로, 그보다 먼저 더미 값을 채워 넣어 이 문제를 우회한다.
import numpy as _np
try:
    _core_mod = getattr(_np, "_core", None) or getattr(_np, "core", None)
    _umath = getattr(_core_mod, "_multiarray_umath", None) if _core_mod is not None else None
    if _umath is not None and not hasattr(_umath, "_blas_supports_fpe"):
        _umath._blas_supports_fpe = lambda *a, **k: False
except Exception:
    pass

WEIGHTS_DIR = os.path.abspath("./weights")
N_WORKERS = min(4, max(1, os.cpu_count() or 1))   # 채점 환경: Standard 4-Core vCPU
TIME_BUDGET_SEC = 1900   # nbconvert timeout(2400s) 대비 여유(약 500s: 모델 로딩/IO)를 남긴 안전 예산
FAST_WORK_WIDTH = 1200      # 1단계(classical CV) 후보 라인 탐색용 축소 폭
FALLBACK_CANVAS = 550       # 2단계(EasyOCR 전체 파이프라인) 축소 캔버스 크기 (속도 우선)

print(f"N_WORKERS={N_WORKERS}  WEIGHTS_DIR={WEIGHTS_DIR}")


## 2. 설계 배경 (요약)

상품 뒷면 사진에는 소비기한 외에도 제조일자, 바코드, 품목보고번호, 전화번호, 영양성분,
LOT 번호 등 다양한 숫자가 섞여 있다. 또한 CPU 전용/오프라인/2400초 타임아웃이라는 강한
제약이 있어, 단순히 고해상도로 전체 이미지를 딥러닝 OCR에 넣는 방식은 시간 초과 위험이 크다
(자세한 실험 근거는 아키텍처 요약서 PDF 참고). 이에 아래와 같은 2단계 캐스케이드 + 시간 예산
가드레일 구조를 채택했다.

1. **1단계 (classical CV, 초저비용)**: 형태학적 연산으로 텍스트 라인 후보 박스를 찾고,
   원본 해상도 그대로 EasyOCR 인식기(recognize)만 호출한다 (검출기 CRAFT는 생략 → 매우 빠름).
   깨끗한 라벨 사진에서는 이 단계만으로 소비기한을 정확히 찾아낸다.
2. **2단계 (fallback, EasyOCR 전체 파이프라인)**: 1단계에서 유효한 날짜 후보를 찾지 못하면,
   대비를 높인(CLAHE) 축소 이미지에 대해 EasyOCR 검출+인식을 전체 실행한다.
3. **키워드 기반 후보 스코어링**: 정규식으로 날짜 형식 후보를 모두 찾은 뒤, 인접 텍스트에
   `소비기한/유통기한/까지` 가 있으면 가점, `제조일자/제조/LOT` 가 있으면 감점하여 최종 날짜를 선택한다.
4. **시간 예산 가드레일**: 전체 처리 시간이 예산을 초과할 것으로 판단되면 남은 이미지는
   즉시 `NONE` 처리하여 채점 타임아웃(정량 0점)을 절대 발생시키지 않도록 한다.


## 3. 날짜 후보 파서 (정규식 + 키워드 스코어링)

In [ ]:
POS_KEYWORDS = ["소비기한", "유통기한", "품질유지기한", "소비 기한", "유통 기한"]
SUFFIX_KEYWORDS = ["까지", "EXP", "exp"]
NEG_KEYWORDS = ["제조일자", "제조일", "제조년월일", "생산일자", "MFG", "mfg", "제조"]
LOT_KEYWORDS = ["LOT", "Lot", "lot", "로트"]

DATE_RE = re.compile(
    r"(?<!\d)(20\d{2})\s*[.\-/,년]\s*(\d{1,2})\s*[.\-/,월]\s*(\d{1,2})\s*일?(?!\d)"
)
DATE_RE_COMPACT = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")


def _valid_ymd(y, m, d):
    y, m, d = int(y), int(m), int(d)
    if not (2020 <= y <= 2035):
        return False
    if not (1 <= m <= 12):
        return False
    if not (1 <= d <= 31):
        return False
    return True


def find_date_candidates(text):
    out = []
    for m in DATE_RE.finditer(text):
        y, mo, d = m.group(1), m.group(2), m.group(3)
        if _valid_ymd(y, mo, d):
            out.append((y, mo.zfill(2), d.zfill(2)))
    if not out:
        for m in DATE_RE_COMPACT.finditer(text):
            y, mo, d = m.group(1), m.group(2), m.group(3)
            if _valid_ymd(y, mo, d):
                out.append((y, mo.zfill(2), d.zfill(2)))
    return out


def _cy(box):
    ys = [p[1] for p in box]
    return sum(ys) / len(ys)


def _cx(box):
    xs = [p[0] for p in box]
    return sum(xs) / len(xs)


def _nearby(box_a, box_b, y_thresh_ratio=1.6):
    ya, yb = _cy(box_a), _cy(box_b)
    ha = max(p[1] for p in box_a) - min(p[1] for p in box_a)
    hb = max(p[1] for p in box_b) - min(p[1] for p in box_b)
    h = max(ha, hb, 1)
    return abs(ya - yb) < h * y_thresh_ratio


def stitch_fragments(ocr_results, x_gap_ratio=2.5, y_overlap_ratio=0.6):
    """같은 줄에서 쪼개진 조각들(예: 2022 / 02.14)을 이어붙여 추가 후보 문자열을 만든다."""
    items = list(ocr_results)
    items.sort(key=lambda t: (_cy(t[0]), _cx(t[0])))
    stitched = []
    used = [False] * len(items)
    for i in range(len(items)):
        if used[i]:
            continue
        group = [items[i]]
        used[i] = True
        cy = _cy(items[i][0])
        ch = max(p[1] for p in items[i][0]) - min(p[1] for p in items[i][0])
        last_xmax = max(p[0] for p in items[i][0])
        for j in range(i + 1, len(items)):
            if used[j]:
                continue
            bj = items[j][0]
            cyj = _cy(bj)
            xminj = min(p[0] for p in bj)
            if abs(cyj - cy) < max(ch, 1) * y_overlap_ratio and 0 <= (xminj - last_xmax) < ch * x_gap_ratio:
                group.append(items[j])
                used[j] = True
                last_xmax = max(p[0] for p in bj)
        if len(group) > 1:
            merged_text = " ".join(g[1] for g in group)
            stitched.append((group[0][0], merged_text, min(g[2] for g in group)))
    return stitched


def extract_best_date(ocr_results):
    """ocr_results: [(box[[x,y]x4], text, conf), ...] -> (year, month, day) or (None, None, None)"""
    if not ocr_results:
        return None, None, None
    all_items = list(ocr_results) + stitch_fragments(ocr_results)
    candidates = []
    for (box, text, conf) in all_items:
        dates = find_date_candidates(text)
        if not dates:
            continue
        context_texts = [text]
        for (box2, text2, conf2) in ocr_results:
            if box2 is box:
                continue
            if _nearby(box, box2):
                context_texts.append(text2)
        context = " ".join(context_texts)
        has_pos = any(k in context for k in POS_KEYWORDS)
        has_suffix = any(k in context for k in SUFFIX_KEYWORDS)
        has_neg = any(k in context for k in NEG_KEYWORDS)
        has_lot = any(k in text for k in LOT_KEYWORDS)
        score = 10.0
        if has_pos:
            score += 100
        if has_suffix:
            score += 60
        if has_neg:
            score -= 90
        if has_lot:
            score -= 40
        score += conf * 5
        for (y, mo, d) in dates:
            candidates.append((score, conf, y, mo, d))
    if not candidates:
        return None, None, None
    candidates.sort(key=lambda t: (-t[0], -t[1]))
    _, _, y, mo, d = candidates[0]
    return y, mo, d


## 4. 1단계: Classical CV 텍스트 라인 후보 검출 (검출기 생략, 초저비용)

In [ ]:
import cv2
import numpy as np


def propose_line_boxes(img_bgr, work_width=FAST_WORK_WIDTH, max_width_ratio=0.30,
                        min_width_ratio=0.02, max_boxes=34):
    h0, w0 = img_bgr.shape[:2]
    scale = work_width / w0
    small = cv2.resize(img_bgr, (work_width, max(1, int(h0 * scale))))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    grad = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    _, bw = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    # 세로 방향으로는 거의 붙이지 않아(커널 높이=1) 서로 다른 줄이 하나로 합쳐지는 것을 방지
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    connected = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    connected = cv2.dilate(connected, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 2)), iterations=1)
    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H, W = small.shape[:2]
    cand = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w < 18 or h < 8:
            continue
        ar = w / float(h)
        if ar < 1.3:
            continue
        if h > H * 0.15:
            continue
        wratio = w / W
        if wratio > max_width_ratio or wratio < min_width_ratio:
            continue   # 문단형 긴 줄(설명문)과 잡음 조각을 제외
        cand.append((w * h, x, y, w, h))
    cand.sort(key=lambda t: -t[0])
    cand = cand[:max_boxes]
    boxes = []
    for area, x, y, w, h in cand:
        boxes.append((x / scale, y / scale, (x + w) / scale, (y + h) / scale))
    return boxes


## 5. OCR 파이프라인 (1단계 fast-path + 2단계 fallback)

In [ ]:
def _to_recognize_format(boxes):
    return [[int(x0), int(x1), int(y0), int(y1)] for (x0, y0, x1, y1) in boxes]


def run_fast_path(reader, img_bgr, gray):
    boxes = propose_line_boxes(img_bgr)
    if not boxes:
        return []
    hlist = _to_recognize_format(boxes)
    try:
        raw = reader.recognize(gray, horizontal_list=hlist, free_list=[])
    except Exception:
        return []
    return list(raw)


def run_fallback(reader, img_bgr, canvas=FALLBACK_CANVAS):
    h0, w0 = img_bgr.shape[:2]
    scale = canvas / max(h0, w0)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    small = cv2.resize(enhanced, (max(1, int(w0 * scale)), max(1, int(h0 * scale))))
    small_bgr = cv2.cvtColor(small, cv2.COLOR_GRAY2BGR)
    try:
        raw = reader.readtext(small_bgr, canvas_size=canvas)
    except Exception:
        return []
    results = []
    for (box, text, conf) in raw:
        scaled_box = [[p[0] / scale, p[1] / scale] for p in box]
        results.append((scaled_box, text, conf))
    return results


def process_one_image(reader, path):
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None, None, None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    fast_results = run_fast_path(reader, img_bgr, gray)
    y, mo, d = extract_best_date(fast_results)
    if y is None:
        fb_results = run_fallback(reader, img_bgr)
        y, mo, d = extract_best_date(fast_results + fb_results)
    return y, mo, d


## 6. 병렬 처리 + 시간 예산 가드레일

`ThreadPoolExecutor` 로 이미지 단위 작업을 워커 스레드에 분배한다. 각 스레드는 최초 호출 시
자신만의 EasyOCR Reader 를 한 번만 로드해 재사용하고(스레드-로컬), 공유 시작 시각을 참고하여
전체 경과 시간이 예산(`TIME_BUDGET_SEC`)을 초과하면 이후 할당된 이미지는 즉시 `NONE` 으로
처리해 절대 타임아웃이 발생하지 않도록 한다. EasyOCR/torch/OpenCV 의 실제 연산 대부분은
C/C++ 구현으로 GIL 을 해제하므로, 스레드 기반으로도 프로세스 기반과 유사한 병렬 처리량을
얻으면서 `fork()` 를 아예 사용하지 않아 Jupyter 커널 환경에서의 안정성을 높인다.


In [ ]:
_thread_local = threading.local()


def _get_worker_reader():
    if not hasattr(_thread_local, "reader"):
        import easyocr
        try:
            import torch
            torch.set_num_threads(1)
        except Exception:
            pass
        _thread_local.reader = easyocr.Reader(
            ["ko", "en"], gpu=False,
            model_storage_directory=WEIGHTS_DIR,
            download_enabled=False,
            verbose=False,
        )
    return _thread_local.reader


def _worker_process(path, start_time, budget):
    image_id = os.path.splitext(os.path.basename(path))[0]
    elapsed = time.time() - start_time
    if elapsed > budget:
        # 예산 초과: 남은 이미지는 안전하게 NONE 처리 (타임아웃 방지 최우선)
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}
    try:
        reader = _get_worker_reader()
        y, mo, d = process_one_image(reader, path)
    except Exception:
        y, mo, d = None, None, None
    if y is None:
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}
    return {"image_id": image_id, "year": y, "month": mo, "day": d, "final_date": f"{y}-{mo}-{d}"}


## 7. 실행: INPUT_DIR 의 모든 이미지에 대해 추론

In [ ]:
image_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.jpg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.jpeg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.png")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPEG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.PNG"))
)
print(f"input images: {len(image_files)}")

results = []
if len(image_files) == 0:
    print("경고: INPUT_DIR 에서 이미지를 찾지 못했습니다.")
else:
    start_time = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(_worker_process, p, start_time, TIME_BUDGET_SEC) for p in image_files]
        for i, fut in enumerate(futures):
            results.append(fut.result())
            if (i + 1) % 200 == 0:
                print(f"  processed {i+1}/{len(image_files)}  elapsed={time.time()-start_time:.1f}s")
    print(f"done. total elapsed={time.time()-start_time:.1f}s")


## 8. submission.csv 저장

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
print(df.head())
print("NONE ratio:", (df["final_date"] == "NONE").mean())
